# Diagnose: PRS conflict crash in `build_germline_data_df`

`pipelines.preprocessing.generate_all_non_text_covariates.build_germline_data_df` raised:

```
ValueError: PRS values disagree across sample IDs for 1528 patient(s).
```

That check (`generate_all_non_text_covariates.py`, `build_germline_data_df`) requires that
when multiple PROFILE sample IDs map to the same `DFCI_MRN`, their PGS values must agree
before collapsing to one row per patient — otherwise a random row choice could hide an
upstream identity error and duplicate a patient across model folds. 1528 patients failing
that check is large enough to investigate before deciding whether to fix the join, fix the
idmap, or relax the check.

This notebook is read-only — no writes anywhere — and distinguishes two very different
explanations:

1. **Fabricated conflict**: `PROFILE_2024_idmap.csv` maps a single `cbio_sample_id` to
   multiple `DFCI_MRN`, so the join fans out and manufactures cross-patient disagreement
   even when each *sample's* PRS values are internally consistent.
2. **Genuine conflict**: a patient legitimately has ≥2 distinct sample IDs whose PRS values
   actually differ — a real identity or data-quality issue upstream (e.g. a resequenced or
   relabeled sample).

**Run this on the cluster**, in the same conda env the pipeline uses — it needs real
`PROFILE_PATH` / `PRS_MATRIX_FILE` data and `SURV_PATH/cohort_df.parquet` (written by
`01_run_preprocessing.ipynb`'s `build_cohort` step) to already exist.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import polars as pl


def find_v2_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "config.py").is_file() and (candidate / "pipelines").is_dir():
            return candidate
    raise RuntimeError(f"Could not find v2 root from {start}")


V2_ROOT = find_v2_root()
if str(V2_ROOT) not in sys.path:
    sys.path.insert(0, str(V2_ROOT))

from config import PROFILE_PATH, PRS_MATRIX_FILE, SURV_PATH

print(f"v2 root:     {V2_ROOT}")
print(f"PROFILE_PATH: {PROFILE_PATH}")
print(f"PRS_MATRIX_FILE: {PRS_MATRIX_FILE}")

## 1. Load cohort and idmap, mirroring `build_germline_data_df` exactly

In [ ]:
cohort_df = pl.read_parquet(os.path.join(SURV_PATH, "cohort_df.parquet"))
print(f"cohort_df: {cohort_df.height} patients")

idmap = pl.read_csv(
    os.path.join(PROFILE_PATH, "PROFILE_2024_idmap.csv"),
    columns=["DFCI_MRN", "cbio_sample_id"],
).with_columns(
    pl.col("DFCI_MRN").cast(pl.Int64, strict=False),
    pl.col("cbio_sample_id").cast(pl.String),
)
cohort_idmap = idmap.filter(pl.col("DFCI_MRN").is_in(cohort_df.get_column("DFCI_MRN")))

print(f"idmap rows total:  {idmap.height}")
print(f"cohort_idmap rows: {cohort_idmap.height}")

## 1b. Full column list of both source files

Only `DFCI_MRN`/`cbio_sample_id` are read from the idmap today, and the join only uses `IID`
+ the PGS columns from the PRS matrix — neither file's full schema has been inspected yet.
If a "most recent sample wins" resolution is wanted for §7's conflicts, the column that
actually encodes recency (a report/collection/sequencing date, a processing batch, a version
tag) needs to be identified here first — do not assume `cbio_sample_id` sorts chronologically
without checking a few real values below.

In [ ]:
idmap_full = pl.read_csv(os.path.join(PROFILE_PATH, "PROFILE_2024_idmap.csv"), n_rows=5)
print("PROFILE_2024_idmap.csv columns:")
for c, dt in idmap_full.schema.items():
    print(f"  {c}: {dt}")
display(idmap_full)

print()
prs_full = pl.read_csv(PRS_MATRIX_FILE, separator="\t", n_rows=5)
print("PRS_MATRIX_FILE columns (first 20):")
for c, dt in list(prs_full.schema.items())[:20]:
    print(f"  {c}: {dt}")
print(f"  ... {len(prs_full.columns)} columns total")
display(prs_full.select(prs_full.columns[:8]))

## 2. Check for idmap fan-out (fabricated-conflict hypothesis)

If one `cbio_sample_id` maps to multiple `DFCI_MRN` here, the join below will attach that
sample'''s single set of PRS values to more than one patient — which can manufacture a
disagreement across patients even though the sample itself never had conflicting values.

In [ ]:
dup_sample = (
    cohort_idmap.group_by("cbio_sample_id")
    .agg(pl.col("DFCI_MRN").n_unique().alias("n_mrn"))
    .filter(pl.col("n_mrn") > 1)
)
print(f"cbio_sample_id values mapping to >1 DFCI_MRN in idmap: {dup_sample.height}")
if dup_sample.height:
    display(dup_sample.head(10))

## 3. Check for duplicate sample rows in the PRS matrix itself

A duplicated `IID` row (e.g. a reprocessed or reloaded sample) with different values would
also manufacture a same-sample disagreement independent of the idmap.

In [ ]:
raw = pl.read_csv(PRS_MATRIX_FILE, separator="\t")
pgs_cols_raw = [c for c in raw.columns if "PGS" in c]

dup_iid = raw.group_by("IID").len().filter(pl.col("len") > 1)
print(f"duplicate IID rows in PRS_MATRIX_FILE itself: {dup_iid.height}")
if dup_iid.height:
    display(dup_iid.head(10))
    example_iid = dup_iid.get_column("IID")[0]
    print(f"\nExample duplicated IID={example_iid}:")
    display(raw.filter(pl.col("IID") == example_iid).select(["IID"] + pgs_cols_raw[:5]))

## 4. Reproduce the join and the conflict check

In [ ]:
joined = (
    raw.rename({"IID": "cbio_sample_id"})
    .with_columns(pl.col("cbio_sample_id").cast(pl.String))
    .join(cohort_idmap, on="cbio_sample_id", how="inner")
)
pgs_cols = [c for c in joined.columns if "PGS" in c]
print(f"joined rows: {joined.height}, {len(pgs_cols)} PGS columns")

conflicts = joined.group_by("DFCI_MRN").agg(
    [pl.col(c).drop_nulls().n_unique().alias(c) for c in pgs_cols]
).filter(pl.max_horizontal(pgs_cols) > 1)
print(f"conflicting patients: {conflicts.height}")

## 5. Samples-per-patient: all joined patients vs. conflicting patients

If conflicts cluster among patients with unusually many mapped samples, that points at the
idmap (more chances to attach an unrelated sample) rather than a data-quality problem
specific to those patients.

In [ ]:
samples_per_mrn = joined.group_by("DFCI_MRN").agg(pl.col("cbio_sample_id").n_unique().alias("n_samples"))
print("samples-per-patient distribution among ALL joined patients:")
display(samples_per_mrn.get_column("n_samples").value_counts().sort("n_samples"))

if conflicts.height:
    conflicting_mrns = conflicts.get_column("DFCI_MRN")
    sub = joined.filter(pl.col("DFCI_MRN").is_in(conflicting_mrns))
    n_samples_conflicting = sub.group_by("DFCI_MRN").agg(pl.col("cbio_sample_id").n_unique().alias("n"))
    print(f"\nsamples-per-patient among the {conflicts.height} CONFLICTING patients:")
    display(n_samples_conflicting.get_column("n").value_counts().sort("n"))

## 6. Inspect one conflicting patient's actual rows

In [ ]:
if conflicts.height:
    one_conflict_mrn = conflicting_mrns[0]
    print(f"Example conflicting patient DFCI_MRN={one_conflict_mrn}, all rows:")
    example = sub.filter(pl.col("DFCI_MRN") == one_conflict_mrn).select(
        ["DFCI_MRN", "cbio_sample_id"] + pgs_cols[:5]
    )
    display(example)
else:
    print("No conflicts found -- nothing to inspect.")

## 7. Confirm conflicts are genuine multi-value disagreements

`drop_nulls().n_unique() > 1` already requires ≥2 *distinct non-null* values per patient, so
a null-vs-value case should not trigger this on its own — confirm that holds for real, since
a bug here would misattribute a null-handling issue as a genuine PRS disagreement.

## 8. Proposed fix: resolve conflicts by taking the most recent sample (BLOCKED)

Decision: when a patient has multiple sample IDs with disagreeing PGS values, keep the
values from the **most recent** sample rather than raising.

**Blocked on identifying the actual recency column** — §1b above dumps the full schema of
both source files; find the column there (report date / collection date / sequencing date /
similar) and put its name in `RECENCY_COL` below. Do **not** assume `cbio_sample_id` sorts
chronologically — confirm it against real date values in §1b first, or this "fix" silently
picks the wrong sample.

Once `RECENCY_COL` is set and this cell is confirmed correct against the conflicting-patient
example in §6, the equivalent change belongs in `build_germline_data_df`
(`generate_all_non_text_covariates.py`): replace the current
`raise ValueError(...)` in the conflict-check block with a sort on `RECENCY_COL` (descending)
before the existing `.group_by("DFCI_MRN", maintain_order=True).agg(pl.all().first())`
collapse, dropping the `conflicts.height` raise. That is a real pipeline behavior change and
should be a deliberate, separate edit — not made inside this diagnostics notebook.

In [ ]:
RECENCY_COL = None  # TODO: set from the §1b schema dump before running this cell

if RECENCY_COL is None:
    print("RECENCY_COL not set -- inspect §1b's column list first, then set it above.")
else:
    resolved = (
        joined.sort(["DFCI_MRN", RECENCY_COL], descending=[False, True])
        .group_by("DFCI_MRN", maintain_order=True)
        .agg(pl.all().first())
    )
    print(f"resolved: {resolved.height} rows (was {joined.get_column('DFCI_MRN').n_unique()} unique patients pre-resolution)")

    # Sanity check against the example conflicting patient from §6.
    if conflicts.height:
        check_mrn = conflicting_mrns[0]
        print(f"\nDFCI_MRN={check_mrn} chosen row after most-recent resolution:")
        display(resolved.filter(pl.col("DFCI_MRN") == check_mrn).select(["DFCI_MRN", "cbio_sample_id", RECENCY_COL] + pgs_cols[:5]))
        print("all candidate rows for this patient, for comparison:")
        display(sub.filter(pl.col("DFCI_MRN") == check_mrn).select(["DFCI_MRN", "cbio_sample_id", RECENCY_COL] + pgs_cols[:5]))

In [ ]:
if conflicts.height:
    first_conflict_col = pgs_cols[0]
    per_patient_vals = (
        sub.group_by("DFCI_MRN")
        .agg(pl.col(first_conflict_col).drop_nulls().unique().alias("vals"))
        .filter(pl.col("vals").list.len() > 1)
    )
    print(f"patients with >1 distinct non-null value for {first_conflict_col}: {per_patient_vals.height}")
    display(per_patient_vals.head(5))

## Summary

Fill in after running:

- idmap fan-out (§2): **?** rows
- duplicate PRS matrix IIDs (§3): **?** rows
- conflicting patients (§4): **?** / cohort
- conflicts cluster among high-sample-count patients (§5)?: **?**
- genuine multi-value disagreement confirmed (§7)?: **?**

This notebook is diagnostic only — it does not change `build_germline_data_df` or the
conflict-check behavior. Any fix (relaxing the check, fixing the idmap, or fixing the join)
is a separate, deliberate change once the root cause here is clear.